**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Random Matrix Theory

What do the eigenvalues of a *random* matrix look like? Not random at all — they obey laws as sharp as the CLT, and those laws decide when [covariance estimation](../../Intro_DSP/Statistical_Signal_Processing.ipynb), [MUSIC](../../Intro_DSP/Array_Processing.ipynb), and PCA can be trusted. Three sessions: the semicircle, Marchenko–Pastur (with the analytic edges verified), and the spiked-model detection threshold — the phase transition every array processor should know by heart.

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S4; [Concentration](../Concentration/Concentration_Inequalities.ipynb) for the 'why so deterministic' intuition.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *The Semicircle Law* (~35 min)
**Goal:** eigenvalues of symmetric random matrices: individually random, collectively deterministic.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (Marchenko–Pastur).

---

## 2. Order from Chaos

💡 **Intuition.** Fill a symmetric matrix with i.i.d. noise, scale by $1/\sqrt{n}$, and its eigenvalue *histogram* converges to a fixed shape — Wigner's semicircle — with no randomness left in the limit. Same magic as the [LLN](../Analysis/Independence.ipynb): each eigenvalue depends on *all* $n^2/2$ entries, no single entry matters ([McDiarmid!](../Concentration/Concentration_Inequalities.ipynb)), so the ensemble self-averages. Eigenvalues also *repel* each other — near-collisions are rare — which is why the histogram is smooth, not clumpy.

In [2]:
n_dim = 2000
M = rng.standard_normal((n_dim, n_dim))
W = (M + M.T) / np.sqrt(2 * n_dim)
eigs = np.linalg.eigvalsh(W)

x = np.linspace(-2.2, 2.2, 300)
semicircle = np.where(np.abs(x) <= 2, np.sqrt(np.maximum(4 - x**2, 0)) / (2*np.pi), 0)
plt.figure(figsize=(7.5, 2.8))
plt.hist(eigs, bins=80, density=True, alpha=0.6, label=f"one {n_dim}×{n_dim} draw")
plt.plot(x, semicircle, "k", linewidth=2, label="Wigner semicircle (exact limit)")
plt.legend(); plt.title("ONE random matrix, and the histogram is already the law")
plt.tight_layout(); plt.show()
print(f"edge check: max eigenvalue {eigs.max():.3f} (limit: 2.0);  fraction outside [−2,2]: {(np.abs(eigs)>2).mean():.4f}")

edge check: max eigenvalue 1.993 (limit: 2.0);  fraction outside [−2,2]: 0.0000


/tmp/ipykernel_2981573/4081657132.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Marchenko–Pastur: the Law of Sample Covariance* (~40 min)
**Goal:** what eigenvalues of pure-noise covariance look like — and why high-dimensional PCA lies.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (spiked models).

---

## 3. The Noise Bulk

💡 **Intuition.** Estimate a covariance from $n$ samples of $p$-dimensional *white* noise (true covariance $= I$: all eigenvalues 1). With $p/n = \gamma$ not small, the sample eigenvalues **spread** across $[(1-\sqrt\gamma)^2, (1+\sqrt\gamma)^2]$ — the Marchenko–Pastur bulk. At $p/n = 1/2$, 'eigenvalues' of pure noise range from 0.09 to 2.9! Every PCA scree plot with $p \sim n$ contains this artifact, and everything inside the bulk is *structurally indistinguishable from noise*.

In [3]:
# ORACLE: empirical bulk edges vs the analytic (1 ± √γ)²   [γ = p/n]
p, n_samp = 1000, 2000
gamma = p / n_samp
X = rng.standard_normal((n_samp, p))
S = X.T @ X / n_samp
eigs = np.linalg.eigvalsh(S)

lo, hi = (1 - np.sqrt(gamma))**2, (1 + np.sqrt(gamma))**2
x = np.linspace(lo, hi, 400)
mp = np.sqrt(np.maximum((hi - x) * (x - lo), 0)) / (2*np.pi*gamma*x)
plt.figure(figsize=(7.5, 2.8))
plt.hist(eigs, bins=80, density=True, alpha=0.6, label="sample covariance of PURE NOISE")
plt.plot(x, mp, "k", linewidth=2, label="Marchenko–Pastur density")
for e in (lo, hi): plt.axvline(e, color="r", linestyle=":", linewidth=1)
plt.legend(); plt.title(f"true covariance = I, yet eigenvalues fill [{lo:.2f}, {hi:.2f}]")
plt.tight_layout(); plt.show()
print(f"analytic edges ({lo:.4f}, {hi:.4f})   empirical (min, max) = ({eigs.min():.4f}, {eigs.max():.4f})")
assert abs(eigs.max() - hi) < 0.05 and abs(eigs.min() - lo) < 0.05

analytic edges (0.0858, 2.9142)   empirical (min, max) = (0.0871, 2.9266)


/tmp/ipykernel_2981573/321660279.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Spiked Models & the Detection Threshold* (~40 min)
**Goal:** when does a real signal's eigenvalue escape the noise bulk? The BBP phase transition.
**Builds on:** Session 2.

---

## 4. The Phase Transition

💡 **Intuition.** Add one rank-one signal of strength $\theta$ to the noise (a source hitting an [array](../../Intro_DSP/Array_Processing.ipynb)). Does the top sample eigenvalue reveal it? **Only above a threshold**: for $\theta > \sqrt{\gamma}$ the top eigenvalue pops out of the bulk at $(1+\theta)(1+\gamma/\theta)$; below it, the spike is *swallowed* — no eigenvalue method can see it, however clever (the BBP transition). This is the sharp version of 'how many snapshots do I need': MUSIC's source count, PCA's component count, all gated by $\theta \gtrless \sqrt{p/n}$.

In [4]:
# sweep the spike strength through the threshold — ORACLE: the BBP position formula
p, n_samp = 400, 1600
gamma = p / n_samp                                     # √γ = 0.5
u = rng.standard_normal(p); u /= np.linalg.norm(u)
thetas = np.linspace(0.05, 1.6, 25)
top_eigs, overlaps = [], []
for theta in thetas:
    X = rng.standard_normal((n_samp, p)) @ np.linalg.cholesky(np.eye(p) + theta*np.outer(u, u)).T
    S = X.T @ X / n_samp
    w, V = np.linalg.eigh(S)
    top_eigs.append(w[-1]); overlaps.append(abs(V[:, -1] @ u))

bulk_edge = (1 + np.sqrt(gamma))**2
bbp = [(1+t)*(1+gamma/t) if t > np.sqrt(gamma) else bulk_edge for t in thetas]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3))
axes[0].plot(thetas, top_eigs, "o", markersize=4, label="top eigenvalue (empirical)")
axes[0].plot(thetas, bbp, "k-", linewidth=1.2, label="BBP prediction")
axes[0].axvline(np.sqrt(gamma), color="r", linestyle=":", label="threshold √γ")
axes[0].axhline(bulk_edge, color="gray", linestyle=":", linewidth=0.8)
axes[0].legend(fontsize=7); axes[0].set_xlabel("spike strength θ"); axes[0].set_title("eigenvalue escape")
axes[1].plot(thetas, overlaps, "o-", markersize=4)
axes[1].axvline(np.sqrt(gamma), color="r", linestyle=":")
axes[1].set_xlabel("spike strength θ"); axes[1].set_title("eigenvector overlap with truth:\nzero below threshold — not weak, ZERO")
plt.tight_layout(); plt.show()

pred_err = np.abs(np.array(top_eigs)[thetas > 0.7] - np.array(bbp)[thetas > 0.7]).max()
print(f"max |top eig − BBP formula| above threshold: {pred_err:.3f}")
print(f"below threshold, eigenvector overlap ≈ {np.mean([o for t, o in zip(thetas, overlaps) if t < 0.35]):.3f} — the signal is invisible")

max |top eig − BBP formula| above threshold: 0.191
below threshold, eigenvector overlap ≈ 0.034 — the signal is invisible


/tmp/ipykernel_2981573/3987407102.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Random eigenvalues obey deterministic laws; pure noise fills a predictable bulk (edges verified to 2 decimals); and signals are detectable by spectra *only* above $\sqrt{p/n}$ — a phase transition, not a gradual fade. Check every scree plot against Marchenko–Pastur before believing a single 'component'.

---
## Where next

- [Array Processing](../../Intro_DSP/Array_Processing.ipynb) — MUSIC's snapshot budget, now quantitative.
- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — why random sketches capture ranges.
- [Concentration](../Concentration/Concentration_Inequalities.ipynb) — the self-averaging machinery.